# Exact (Hill) satellite on the grid of the satellite-start map (22/10/2026)

Report v2, big error 1, "TO DO for v3": seed `lyapunov_heatmap_satellite_start.ipynb` with the exact harmonic-balance satellite instead of the analytic one (`satellite_hb`).
The analytic seed exists only above an analytic boundary that is 8–15% below the true fold, and between the fold and about 1.2 × the fold it is too far from the satellite (1.1–1.4 in $x^\lambda$ units at 40–50 MHz, $\varepsilon_d = 10.12\,g_2$) and ends on the cat.

**What this notebook does.** For every $\varepsilon_d/g_2$ of the map ($2{:}0.1{:}13.5$) the period-1 satellite is followed by pseudo-arclength from the top of the map ($\varepsilon_p = 1.1$, $g_2/2\pi = 122$ MHz) towards lower $g_2$, down to its **fold** (saddle-node of limit cycles, where it is born together with the saddle, `Hill_method_satellite_fold.ipynb`).
The state at $\tau = 0$ is stored at every $\varepsilon_p$ of the map ($0.2{:}0.01{:}1.1$) above the fold; below the fold the satellite does not exist and the entry is `null`.
The fold sets the lower threshold in $g_2$: steps of the arclength that pass it (the parameter $s$ turns back) are rejected and halved, so the branch is never continued onto the saddle.

**Start.** The 60 MHz satellite at $\varepsilon_d = 10.12\,g_2$ (converged from the analytic seed, as in `Hill_method_satellite_fold`), continued in $g_2$ up to 122 MHz, then in $\varepsilon_d/g_2$ along the top of the map.
It is the member $\sigma = +1$ of the pair $a \to -a$, the same member as the old analytic seed.

**Conventions** (as in the other Hill notebooks): $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\varepsilon_d = (\varepsilon_d/g_2)\,g_2$, $\kappa_b = \pi/10.4$ (code), $\tau = \omega_a t$, $x^\lambda = \varphi x$, $N_H = 16$.

**Output** `data/hill_orbits/satellite_seed_hill_22_10_2026.json`: `state[j][i]` $= [x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ at $\tau = 0$ for ratio index `j` and $\varepsilon_p$ index `i`, the four Floquet multipliers (`max_abs_mu`, leading multiplier), and the fold `g2_fold_MHz[j]`.
In Julia (`SystemATS`, $t = 0$): $a = (x_a + i\,dx_a/d\tau)/(\sqrt2\varphi_a)$, $b = (x_b + i\,(dx_b/d\tau)/2)/(\sqrt2\varphi_b)$, then `system_from_amplitudes(a, b)`.
The whole notebook runs headless in about 5 min.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import *                      # w_a, phi_a, phi_b, E_J, kappa_b (pi/10.4, code convention), G0 (g2 = s G0), MHz
from catchaos.plotting import *                    # Plot_setting (figsize, Perso_palette_float, ...)
from catchaos.ats import HB, full_rhs
from catchaos.hill import strobe, memory_phase, monodromy, branch_on_grid
from catchaos.seeds import satellite_hb, seed_z
from catchaos.jsonio import nan2none

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/"

## Harmonic balance (`catchaos`, as in `Hill_method_satellite_fold.ipynb`)
Exact one-period AFT (`catchaos.ats.HB`). `strobe(z, N_H)` is the state $[x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ at $\tau = 0$, `monodromy` the four Floquet multipliers, `full_rhs` the equations of motion in $\tau$ (for the return check).

## Analytic satellite (only used once, to converge the 60 MHz satellite at $\varepsilon_d = 10.12\,g_2$)
`satellite_hb` and `seed_z` come from `catchaos.seeds`.

## Grid of the map, and the satellite along its top edge
Same grid as `lyapunov_heatmap_satellite_start.ipynb`: `vcat(0.2:.01:1.1)` and `2:.1:13.5`.

In [ ]:
eps_p_list = np.round(np.arange(0.2, 1.1 + 1e-9, 0.01), 2)
ratio_list = np.round(np.arange(2.0, 13.5 + 1e-9, 0.1), 1)
s_grid = np.sin(eps_p_list)
s_top = s_grid[-1]

N_H = 16
r0, g_seed = 10.12, 60.0
hb = HB(N_H)
s_seed = g_seed/MHz(G0)
hb.set_s(s_seed, r0)
z_sat60 = hb.solve(seed_z(satellite_hb(s_seed, r0), N_H))

z = z_sat60.copy()                                      # up to the top of the map at 10.12
for s_new in np.linspace(s_seed, s_top, 25)[1:]:
    hb.set_s(s_new, r0); z = hb.solve(z)
z_top = {}                                              # then along the top edge, in eps_d/g2
for part in (ratio_list[ratio_list > r0], ratio_list[ratio_list < r0][::-1]):
    zz, r_prev = z.copy(), r0
    for r in part:
        for rr in np.linspace(r_prev, r, 4)[1:]:
            hb.set_s(s_top, rr); zz = hb.solve(zz)
        z_top[r] = zz.copy(); r_prev = r
ph = np.array([memory_phase(z_top[r]) for r in ratio_list])
print(f"satellite at g2/2pi = {MHz(s_top*G0):.1f} MHz for all {len(ratio_list)} ratios; memory phase {ph.min():+.3f} .. {ph.max():+.3f} pi")

## Continuation down to the fold, at every $\varepsilon_d/g_2$
`satellite_column` (`catchaos.hill.branch_on_grid(..., direction=-1)`) runs the arclength in $(z, s)$ towards lower $s$. Each grid value of $s$ crossed by a step is solved at fixed $s$ from the linear interpolation between the two ends of the step (and must stay within 1% of it).
A step after which $s$ increases has passed the fold: it is rejected and $ds$ halved, until $ds < 10^{-6}$. The last accepted point gives the fold.
The Floquet multipliers are then computed at every grid point (monodromy matrix).

In [ ]:
n_r, n_p = len(ratio_list), len(eps_p_list)
state = np.full((n_r, n_p, 4), np.nan)        # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = 0
mu_all = np.full((n_r, n_p, 4), np.nan + 0j)
s_fold = np.full(n_r, np.nan)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # overflows inside rejected Powell iterations
    for j, r in enumerate(ratio_list):
        zs, s_fold[j], _ = branch_on_grid(hb, lambda s: hb.set_s(s, r), z_top[r], s_top, s_grid, direction=-1)
        for i in np.where(np.isfinite(zs[:, 0]))[0]:
            state[j, i] = strobe(zs[i], N_H)
            hb.set_s(s_grid[i], r); mu_all[j, i] = monodromy(hb, zs[i])
g2_fold = MHz(s_fold*G0)
max_abs_mu = np.abs(mu_all).max(axis=2)
mu_lead = np.take_along_axis(mu_all, np.nanargmax(np.abs(np.where(np.isnan(mu_all), 0, mu_all)), axis=2)[..., None], axis=2)[..., 0]
has = np.isfinite(state[..., 0])
print(f"{time.time() - t0:.0f} s; satellite on {has.sum()}/{has.size} grid points ({100*has.mean():.1f}%), "
      f"stable (max|mu| < 1) on {(max_abs_mu < 1).sum()}")
print(f"satellite fold: {g2_fold.max():.2f} MHz (eps_d/g2 = {ratio_list[0]}) .. {g2_fold.min():.2f} MHz (eps_d/g2 = {ratio_list[-1]})")

## Checks
1. The fold against `satellite_fold_line_20_10_2026.json` (`Hill_method_satellite_fold`, ratios $2{:}0.5{:}13.5$, on this grid).
2. The lowest grid point of each column is the satellite, not the saddle: no real multiplier above 1 there.
3. The stored state is on the orbit: time integration of the full equations over one period from it, at the lowest grid point of each column.
4. Stability along $\varepsilon_d = 10.1\,g_2$: the satellite should be stable from the fold (40.4 MHz) to its period doubling ($\approx 75$ MHz), and again above $\approx 104$ MHz.

In [ ]:
with open(OUT_DIR + "hill_orbits/satellite_fold_line_20_10_2026.json") as f:
    ref = json.load(f)
ref_r, ref_g = np.array(ref["eps_d_over_g2"]), np.array(ref["g2_fold_MHz_hill"])
mine = np.array([g2_fold[np.argmin(np.abs(ratio_list - r))] for r in ref_r])
print(f"1. fold vs satellite_fold_line_20_10_2026: max |difference| = {np.max(np.abs(mine - ref_g)):.3f} MHz")

i_low = np.argmax(has, axis=1)                         # lowest eps_p with a satellite, per ratio
mu_low = mu_all[np.arange(n_r), i_low]
real_above_1 = np.any((np.abs(mu_low.imag) < 1e-9) & (mu_low.real > 1), axis=1)
print(f"2. lowest grid point above the fold: max|mu| = {np.abs(mu_low).max(axis=1).min():.3f} .. {np.abs(mu_low).max(axis=1).max():.3f}, "
      f"real multiplier > 1 (saddle) at {real_above_1.sum()} ratios; gap to the fold {np.min(MHz(s_grid[i_low]*G0) - g2_fold):.2f} .. {np.max(MHz(s_grid[i_low]*G0) - g2_fold):.2f} MHz")

err = []
for j, r in enumerate(ratio_list):
    y0 = state[j, i_low[j]]
    y1 = solve_ivp(full_rhs(s_grid[i_low[j]], r), (0, 2*np.pi), y0, method="DOP853", rtol=1e-11, atol=1e-13).y[:, -1]
    err.append(np.linalg.norm(y1 - y0)/np.linalg.norm(y0))
print(f"3. one-period return error, relative: max {np.max(err):.1e}")

j = np.argmin(np.abs(ratio_list - 10.1))
st = max_abs_mu[j] < 1
edges = np.where(np.diff(st.astype(int)) != 0)[0]
print(f"4. eps_d/g2 = {ratio_list[j]}: stability changes between g2/2pi = " +
      ", ".join(f"{MHz(s_grid[e]*G0):.1f}-{MHz(s_grid[e + 1]*G0):.1f} MHz" for e in edges) + f" (fold at {g2_fold[j]:.2f} MHz)")

In [ ]:
g2_MHz_list = MHz(s_grid*G0)
alpha_sq = ratio_list/2
fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
amp = np.where(has, np.hypot(state[..., 0], state[..., 2])/(np.sqrt(2)*phi_a), np.nan)   # |a(0)|
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, amp, cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$|a(0)|$")
ax.set_title("Seed: exact satellite at $t = 0$", fontsize=8)
ax = axes[1]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, np.log10(max_abs_mu), cmap="RdBu_r", vmin=-0.5, vmax=0.5, shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$\log_{10}\max|\mu|$")
ax.contour(g2_MHz_list, alpha_sq, np.where(has, max_abs_mu, 0.5), levels=[1], colors="k", linewidths=0.6)
ax.set_title("Floquet stability of the satellite", fontsize=8)
for ax in axes:
    ax.plot(g2_fold, alpha_sq, "k--", lw=1, label="satellite fold (Hill)")
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
    ax.set_xlim(g2_MHz_list.min(), g2_MHz_list.max()); ax.set_ylim(alpha_sq.min(), alpha_sq.max())
axes[0].legend(fontsize=5, loc="upper right")
plt.tight_layout()
plt.savefig(FIG_DIR + "satellite_seed_hill_grid_22_10_2026.pdf")
plt.show()

## Export
`null` below the fold. Outer index: $\varepsilon_d/g_2$ (`j`), inner index: $\varepsilon_p$ (`i`), i.e. `state[j][i]`, as `Lyap_coef_chaos[j, i]` in the Julia map.

In [ ]:
with open(OUT_DIR + "hill_orbits/satellite_seed_hill_22_10_2026.json", "w") as f:
    json.dump({"description": "Exact period-1 satellite (harmonic balance, pseudo-arclength from the top of the map down to its fold) "
                              "on the grid of lyapunov_heatmap_satellite_start.ipynb; Hill_satellite_seed_grid.ipynb",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H,
               "state_definition": "[x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = omega_a t = 0, x^lambda = phi x; "
                                   "a = (x_a + i dx_a/dtau)/(sqrt2 phi_a), b = (x_b + i (dx_b/dtau)/2)/(sqrt2 phi_b); null below the fold",
               "eps_p_list": eps_p_list.tolist(), "eps_d_over_g2": ratio_list.tolist(), "g2_MHz": g2_MHz_list.tolist(),
               "state": nan2none(state), "max_abs_mu": nan2none(max_abs_mu),
               "mu_lead_re": nan2none(mu_lead.real), "mu_lead_im": nan2none(mu_lead.imag),
               "s_fold": s_fold.tolist(), "g2_fold_MHz": g2_fold.tolist()}, f)
print("written", OUT_DIR + "hill_orbits/satellite_seed_hill_22_10_2026.json")